### Phase 1 Audit

#### 1.1 Missing value inventory


In [ ]:
import os
import re
import io 
import sys
import csv
import time
import json
import random
import urllib3
import numpy as np
import pandas as pd
import urllib.parse
import urllib.request
from tqdm import tqdm
from time import sleep
from geopy.geocoders import Nominatim
from geopy.distance import great_circle


sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from config import PREPROCESSED, RAW_DATA_DIR, DATA_DIR

COORD_FIELDS = ["LAT", "LON"]

ADDR_FIELDS  = ["STR_NBR", "STR_NBR_NUM", "STR_NM", "STR_TYP_CD", "LOC_ZIP_CD"]
NULL_SENTINELS = {"0.0", "0", "", " ", "  ", "nan", "none", "null", "n/a"}

In [ ]:
# loading all csv files
def is_missing(val):
    if val is None:
        return True
    if isinstance(val, float) and np.isnan(val):
        return True
    return str(val).strip().lower() in NULL_SENTINELS

csv_files = sorted([f for f in os.listdir(RAW_DATA_DIR) if f.lower().endswith(".csv")])
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in: {RAW_DATA_DIR}")

print(f"Found {len(csv_files)} CSV file(s)\n")
frames = []
for fname in csv_files:
    fpath = os.path.join(RAW_DATA_DIR, fname)
    df_year = pd.read_csv(fpath, dtype=str, encoding="latin-1", low_memory=False)
    df_year.columns = df_year.columns.str.strip()

    if "data_year" not in df_year.columns or df_year["data_year"].isna().all():
        m = re.search(r"(20\d{2}|19\d{2})", fname)
        df_year["data_year"] = m.group(1) if m else "unknown"

    frames.append(df_year)
    print(f"{fname} -> {len(df_year):,} rows")

df = pd.concat(frames, ignore_index=True, sort=False)

# strip whitespace from all location fields
for field in COORD_FIELDS + ADDR_FIELDS:
    if field in df.columns:
        df[field] = df[field].astype(str).str.strip()

df["data_year"] = df["data_year"].astype(str).str.strip()
print(f"Total loaded: {len(df):,} rows")

In [ ]:
def flag_missing(series):
    return series.isna() | series.astype(str).str.strip().str.lower().isin(NULL_SENTINELS)

# assigning coordinate status
lat_miss = flag_missing(df.get("LAT"))
lon_miss = flag_missing(df.get("LON"))
df["_coord_status"] = "neither"
df.loc[~lat_miss & ~lon_miss, "_coord_status"] = "both"
df.loc[~lat_miss &  lon_miss, "_coord_status"] = "lat_only"
df.loc[ lat_miss & ~lon_miss, "_coord_status"] = "lon_only"

# assigning address score
# 0-4 integer score:+1  if STR_NBR or STR_NBR_NUM, +1 if STR_NM,+1 if STR_TYP_CD +1 if LOC_ZIP_CD
score = pd.Series(0, index=df.index)
nbr = ~flag_missing(df.get("STR_NBR")) | ~flag_missing(df.get("STR_NBR_NUM"))
score += nbr.astype(int)
for f in ["STR_NM", "STR_TYP_CD", "LOC_ZIP_CD"]:
    if f in df.columns:
        score += (~flag_missing(df[f])).astype(int)
df["_addr_score"]   = score

# assigning buckets
# A = valid coords (both LAT+LON)
# B = no coords, address score >= 3 (strong geocoding candidate),
# C = no coords, address score 1-2 (weak geocoding candidate)
# D = no coords, address score 0 (unresolvable)
cs = df["_coord_status"]
addr_sc = df["_addr_score"]
df["_bucket"] = "D"
has_coords = cs == "both"
df.loc[has_coords, "_bucket"] = "A"
df.loc[~has_coords & (addr_sc >= 3), "_bucket"] = "B"
df.loc[~has_coords & addr_sc.between(1, 2), "_bucket"] = "C"

In [ ]:
rows = []
for year in sorted(df["data_year"].unique()):
    sub = df[df["data_year"] == year]
    n = len(sub)
    cc = sub["_coord_status"].value_counts()
    bc = sub["_bucket"].value_counts()
    sc = sub["_addr_score"].value_counts()

    rows.append({
        "year": year, "total_rows": n,
        "coords_both": cc.get("both",     0),
        "coords_lat_only": cc.get("lat_only", 0),
        "coords_lon_only": cc.get("lon_only", 0),
        "coords_neither": cc.get("neither",  0),
        "pct_coords_ok":  round(cc.get("both", 0) / n * 100, 1),
        **{f"addr_score_{s}": sc.get(s, 0) for s in range(5)},
        **{f"miss_{f}": round(flag_missing(sub[f] if f in sub.columns else pd.Series()).mean() * 100, 1) for f in COORD_FIELDS + ADDR_FIELDS},
        **{f"bucket_{b}": bc.get(b, 0) for b in "ABCD"},
        **{f"pct_{b}":    round(bc.get(b, 0) / n * 100, 1) for b in "ABCD"},
    })
stats = pd.DataFrame(rows)

total_row = {
    "year": "All Combined",
    "total_rows": stats["total_rows"].sum(),
    "coords_both": stats["coords_both"].sum(),
    "coords_lat_only": stats["coords_lat_only"].sum(),
    "coords_lon_only": stats["coords_lon_only"].sum(),
    "coords_neither": stats["coords_neither"].sum(),
    "pct_coords_ok": round((stats["coords_both"].sum() / stats["total_rows"].sum()) * 100, 1)
}
for s in range(5):
    total_row[f"addr_score_{s}"] = stats[f"addr_score_{s}"].sum()
for b in "ABCD":
    total_row[f"bucket_{b}"] = stats[f"bucket_{b}"].sum()
    total_row[f"pct_{b}"] = round((stats[f"bucket_{b}"].sum() / stats["total_rows"].sum()) * 100, 1)
for f in COORD_FIELDS + ADDR_FIELDS:
    weighted_mean = (stats[f"miss_{f}"] * stats["total_rows"]).sum() / stats["total_rows"].sum()
    total_row[f"miss_{f}"] = weighted_mean

stats = pd.concat([stats, pd.DataFrame([total_row])], ignore_index=True)

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
stats

#### 1.2 Coordinate plausibility check


In [ ]:
# Convert LAT/LON to float
df["LAT_f"] = pd.to_numeric(df["LAT"], errors="coerce")
df["LON_f"] = pd.to_numeric(df["LON"], errors="coerce")

BUFFER_DEG = 0.6

# Use only bucket A rows (both coords present) to derive bounding box
bucket_a = df[df["_bucket"] == "A"]
lat_p1, lat_p99 = bucket_a["LAT_f"].quantile([0.01, 0.99])
lon_p1, lon_p99 = bucket_a["LON_f"].quantile([0.01, 0.99])

LAT_MIN = lat_p1 - BUFFER_DEG
LAT_MAX = lat_p99 + BUFFER_DEG
LON_MIN = lon_p1 - BUFFER_DEG
LON_MAX = lon_p99 + BUFFER_DEG

print("1.2 Coordinate plausibility check (Bucket A only):")
print(f"Derived bounding box (1st-99th pct + buffer): LAT : {LAT_MIN:.4f} to {LAT_MAX:.4f}, LON : {LON_MIN:.4f} to {LON_MAX:.4f}")

has_coords = df["_bucket"] == "A"
n_total = len(df)
n_bucket_a = has_coords.sum()

zero_sentinel = ((df["LAT_f"] == 0.0) | (df["LON_f"] == 0.0)) & has_coords
out_of_region = (has_coords & ~zero_sentinel & ((df["LAT_f"] < LAT_MIN) | (df["LAT_f"] > LAT_MAX) | (df["LON_f"] < LON_MIN) | (df["LON_f"] > LON_MAX)))
transposed = (has_coords & ~zero_sentinel & ((df["LON_f"] > 0) | (df["LAT_f"] < 0) | ((df["LAT_f"] < -100) | df["LON_f"].between(30, 50))))

print(f"Zero sentinel (LAT or LON == 0) : {zero_sentinel.sum():,}")
print(f"Outside derived bounding box : {out_of_region.sum():,}")
print(f"Likely transposed coords : {transposed.sum():,}")

any_flag = zero_sentinel | out_of_region | transposed
clean_count = n_bucket_a - any_flag.sum()
print(f"Clean Bucket A rows (no issues): {clean_count:,}")

# Stored coordinates that fail any plausibility check are not used: demote them to the address-based
# buckets so they become geocoding targets (B: score >= 3, C: score 1-2, D: no usable address).
addr = df.loc[any_flag, "_addr_score"]
df.loc[any_flag, "_bucket"] = np.where(addr >= 3, "B", np.where(addr >= 1, "C", "D"))
print(f"Demoted from bucket A (implausible stored coords): {any_flag.sum():,}")

### Phase 2 Coordinate Verification (Bucket A)

In [ ]:
SAMPLE_PER_YEAR = 30
RANDOM_SEED     = 42
NOMINATIM_URL  = "https://nominatim.openstreetmap.org/reverse"
USER_AGENT     = "la-car-theft-research/1.0"
DELAY_SEC      = 1.1   # slightly over 1s

# sampling bucket A rows with good addresses (score >= 2)
sample_pool = df[(df["_bucket"] == "A") & (df["_addr_score"] >= 2)].copy()
random.seed(RANDOM_SEED)

yearly_samples = []
for year in sorted(sample_pool["data_year"].unique()):
    yr_pool = sample_pool[sample_pool["data_year"] == year]
    n = min(SAMPLE_PER_YEAR, len(yr_pool))
    yearly_samples.append(yr_pool.sample(n, random_state=RANDOM_SEED))

sample_df = pd.concat(yearly_samples, ignore_index=True)

def reverse_geocode(lat, lon):
    params = urllib.parse.urlencode({
        "lat": lat, "lon": lon,
        "format": "jsonv2",
        "addressdetails": 1,
        "zoom": 16
    })
    req = urllib.request.Request(f"{NOMINATIM_URL}?{params}", headers={"User-Agent": USER_AGENT})
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            data = json.loads(resp.read().decode())
        addr = data.get("address", {})
        return {
            "rev_road": addr.get("road", ""),
            "rev_house_number": addr.get("house_number", ""),
            "rev_postcode": addr.get("postcode", ""),
            "rev_city": addr.get("city") or addr.get("town") or addr.get("village") or "",
            "rev_display": data.get("display_name", ""),
            "rev_error": ""
        }
    except Exception as e:
        return {"rev_road": "", "rev_house_number": "", "rev_postcode": "", "rev_city": "", "rev_display": "", "rev_error": ""} | {"rev_error": str(e)}

# reverse geocode (cached: the sample is fixed by RANDOM_SEED, so reruns reuse the checkpoint)
REVERSE_CHECKPOINT_FILE = os.path.join(DATA_DIR, "phase2_reverse.csv")
if os.path.exists(REVERSE_CHECKPOINT_FILE):
    rev_df = pd.read_csv(REVERSE_CHECKPOINT_FILE, index_col="_orig_idx", keep_default_na=False)
    print(f"Loaded {len(rev_df):,} cached reverse-geocode results")
else:
    results = []
    for i, row in tqdm(enumerate(sample_df.itertuples(index=True)), total=len(sample_df)):
        rev = reverse_geocode(row.LAT_f, row.LON_f)
        rev["_orig_idx"] = row.Index
        results.append(rev)
        time.sleep(DELAY_SEC)
    rev_df = pd.DataFrame(results).set_index("_orig_idx")
    rev_df.to_csv(REVERSE_CHECKPOINT_FILE)
sample_df = sample_df.join(rev_df)

In [ ]:
#  street matching utilities
def normalize_street(s):
    if not isinstance(s, str):
        return ""
    s = s.lower().strip()
    abbrevs = {
        r"\bav\b":"ave", r"\bave\b":"ave", r"\bblvd\b":"blvd",
        r"\bst\b":"st",  r"\bdr\b":"dr",   r"\bpl\b":"pl",
        r"\brd\b":"rd",  r"\bln\b":"ln",   r"\bct\b":"ct",
        r"\bwy\b":"way", r"\bfwy\b":"fwy", r"\bhwy\b":"hwy",
        r"\bcyn\b":"canyon", r"\bmtn\b":"mountain",
    }
    for pat, repl in abbrevs.items():
        s = re.sub(pat, repl, s)
    return re.sub(r"[^a-z0-9\s]", "", s).strip()

def street_in_display(stored_nm, rev_display):
    a = normalize_street(stored_nm)
    b = normalize_street(rev_display)
    if not a or not b:
        return False
    words = [w for w in a.split() if len(w) > 2]
    return all(w in b for w in words) if words else False

def is_numeric_street(name):
    return bool(re.match(r"^\d+(st|nd|rd|th)?$", str(name).strip().lower()))

def zip_match(stored_zip, rev_postcode):
    a = str(stored_zip).strip()[:5]
    b = str(rev_postcode).strip()[:5]
    return a == b and len(a) == 5

def reclassify_suspect(row):
    stored_nm  = str(row.get("STR_NM",""))
    crs_nm     = str(row.get("CRS_STR_NM",""))
    rev_road   = str(row.get("rev_road",""))
    rev_display= str(row.get("rev_display",""))
    zip_ok     = zip_match(row.get("LOC_ZIP_CD",""), row.get("rev_postcode",""))

    if street_in_display(stored_nm, rev_display):
        return "verified_display"
    if is_numeric_street(stored_nm):
        return "numeric_street"
    if crs_nm and street_in_display(crs_nm, rev_road):
        return "likely_corner"
    if zip_ok:
        return "zip_verified"
    return "suspect"

# apply verdict
valid = sample_df[sample_df["rev_error"] == ""].copy()
valid["coord_verdict"] = valid.apply(
    lambda r: "verified_primary" if street_in_display(r.get("STR_NM",""), r.get("rev_road","")) else reclassify_suspect(r),
    axis=1
)

In [ ]:
n   = len(valid)
order = ["verified_primary","verified_display","numeric_street", "likely_corner","zip_verified","suspect"]
counts = valid["coord_verdict"].value_counts()

print("2.1 Verdict breakdown")
for v in order:
    c = counts.get(v,0)
    print(f"{v}: {c} ({c/n*100:.1f}%)")

trusted = sum(counts.get(v,0) for v in order if v != "suspect")
suspect = counts.get("suspect",0)
rate = trusted / n * 100

print(f"\nTrusted (non-suspect) : {trusted} ({rate:.1f}%)")
print(f"Genuinely suspect : {suspect} ({suspect/n*100:.1f}%)")

### Phase 3 Geocoding

#### 3.0 Viewbox Anchors

In [ ]:
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
CENSUS_URL = "https://geocoding.geo.census.gov/geocoder/locations/addressbatch"
USER_AGENT = "la-car-theft-research/1.0"
CENSUS_CHECKPOINT_FILE = os.path.join(DATA_DIR, "phase3_census.csv")
NOMINATIM_CHECKPOINT_FILE = os.path.join(DATA_DIR, "phase3_nominatim.csv")
CENSUS_BATCH_SIZE = 9000
DELAY_SEC = 1.1

# Viewbox anchors (RPT_DIST → ZIP → full bbox)
clean_a = df[(df["_bucket"] == "A")].copy()

def _rpt_norm(s):
    v = pd.to_numeric(s, errors="coerce")
    return str(int(v)) if pd.notna(v) else None

def _build_centroid_lookup(group_col, df_clean):
    stats = (
        df_clean.groupby(group_col)
        .agg(lat=("LAT_f","median"), lon=("LON_f","median"), n=("LAT_f","count"), lat_std=("LAT_f","std"), lon_std=("LON_f","std"))
        .reset_index()
    )
    stats["spread_m"] = (((stats["lat_std"].fillna(0)*111_000)**2 + (stats["lon_std"].fillna(0)*111_000)**2) ** 0.5)
    reliable = stats[(stats["n"] >= 10) & (stats["spread_m"] < 3000)]
    return dict(zip(reliable[group_col], zip(reliable["lat"], reliable["lon"], reliable["spread_m"])))

# RPT_DIST centroids
clean_a["_rpt"] = clean_a["RPT_DIST_NBR"].apply(_rpt_norm)
clean_a = clean_a[clean_a["_rpt"].notna()]
rpt_centroid = { k: v for k, v in _build_centroid_lookup("_rpt", clean_a).items() if not str(k).startswith("99")}   # exclude administrative codes

# ZIP centroids
clean_a["_zip"] = clean_a["LOC_ZIP_CD"].astype(str).str.strip().str[:5]
clean_a_zip = clean_a[clean_a["_zip"].str.match(r"^\d{5}$")]
zip_centroid = _build_centroid_lookup("_zip", clean_a_zip)

# Normalise keys on full df
df["_rpt"] = df["RPT_DIST_NBR"].apply(_rpt_norm)
df["_zip"] = df["LOC_ZIP_CD"].astype(str).str.strip().str[:5]
df.loc[~df["_zip"].str.match(r"^\d{5}$"), "_zip"] = None

VBOX_RPT = {(0, 500): 0.02, (500, 1500): 0.03, (1500, 3000): 0.05}
VBOX_ZIP = {(0, 500): 0.03, (500, 1500): 0.05, (1500, 3000): 0.07}

def _spread_to_delta(spread_m, table):
    for (lo, hi), delta in table.items():
        if lo <= spread_m < hi:
            return delta
    return max(table.values())

def get_viewbox(rpt, zip_code):
    if rpt and rpt in rpt_centroid:
        lat, lon, spread = rpt_centroid[rpt]
        delta = _spread_to_delta(spread, VBOX_RPT)
        return lat, lon, delta, "rpt"
    z = str(zip_code) if pd.notna(zip_code) else ""
    if z and z in zip_centroid:
        lat, lon, spread = zip_centroid[z]
        delta = _spread_to_delta(spread, VBOX_ZIP)
        return lat, lon, delta, "zip"
    return (LAT_MIN+LAT_MAX)/2, (LON_MIN+LON_MAX)/2, (LAT_MAX-LAT_MIN)/2, "bbox"

print(f"RPT_DIST centroids : {len(rpt_centroid):,}")
print(f"ZIP centroids : {len(zip_centroid):,}")

#### 3.1 Build Address String

In [ ]:
# Build address string
STREET_TYPE_MAP = { "AV":"AVE","BL":"BLVD","WY":"WAY","TR":"TRL","HY":"HWY", "PY":"PKWY","PW":"PKWY","FY":"FWY","EX":"EXPY","CI":"CIR", "AL":"ALY","TL":"TRL","WK":"WALK","PK":"PARK","PA":"PASS"}
KNOWN_JUNK = {"LASD","LASO","CHP","LAPD","LAUSD","LACSD","LACFD","LAWA","NEVADA","ARIZONA","OREGON","HAWAII","LAS VEGAS","SAN DIEGO","LONG BEACH","POMONA","PASADENA","GLENDALE","BURBANK","SANTA ANA","ANAHEIM","RIVERSIDE","POLICE","SHERIFF"}

_NUMBERED = re.compile(r"^\d+(ST|ND|RD|TH)?$", re.I)

def is_junk(nm):
    if not isinstance(nm, str): return True
    s = nm.strip().upper()
    if s in KNOWN_JUNK: return True
    if _NUMBERED.match(s): return False
    if len(s) <= 5 and not any(v in s for v in "AEIOU") and not s.isdigit():
        return True
    return False

def clean_field(s):
    s = str(s).strip()
    return "" if s.lower() in NULL_SENTINELS | {"none","nan"} else s

def build_address(row):
    """
    Returns (street, city, state, zip, pattern) for Census batch format.
    Returns None if no usable address can be built.
    """
    nbr = clean_field(row.get("STR_NBR",""))
    frac = clean_field(row.get("STR_FRAC_NBR",""))
    dirr = clean_field(row.get("STR_DIR_CD",""))
    nm = clean_field(row.get("STR_NM",""))
    typ = STREET_TYPE_MAP.get(clean_field(row.get("STR_TYP_CD","")),clean_field(row.get("STR_TYP_CD","")))
    crs  = clean_field(row.get("CRS_STR_NM",""))
    city = clean_field(row.get("CITY_NM","")) or "Los Angeles"
    zipc = clean_field(row.get("LOC_ZIP_CD",""))[:5]

    if not nm or is_junk(nm):
        return None

    nbr_str    = " ".join(filter(None, [nbr, frac]))
    street_str = " ".join(filter(None, [dirr, nm, typ]))

    if nbr_str:
        return street_str and (f"{nbr_str} {street_str}", city, "CA", zipc, "numbered")
    elif crs and not is_junk(crs):
        crs_str = clean_field(row.get("CRS_STR_NM",""))
        return (f"{street_str} & {crs_str}", city, "CA", zipc, "intersection")
    else:
        return None   # name-only - not worth sending

#### 3.2 Select targets and attach viewbox

In [ ]:
# Select targets and attach viewbox

targets = df[df["_bucket"].isin(["B","C"])].copy()

addr_fields = targets.apply(build_address, axis=1)
targets["_addr_tuple"] = addr_fields
targets = targets[targets["_addr_tuple"].notna()]

targets[["census_street","census_city","census_state",
         "census_zip","_pattern"]] = pd.DataFrame(
    targets["_addr_tuple"].tolist(), index=targets.index
)
targets.drop(columns="_addr_tuple", inplace=True)

vb = targets.apply(
    lambda r: get_viewbox(r["_rpt"], r["_zip"]), axis=1
)
targets["vb_lat"]    = vb.apply(lambda x: x[0])
targets["vb_lon"]    = vb.apply(lambda x: x[1])
targets["vb_delta"]  = vb.apply(lambda x: x[2])
targets["vb_anchor"] = vb.apply(lambda x: x[3])

print(f"Geocoding targets  : {len(targets):,}")
print(f"  numbered         : {(targets['_pattern']=='numbered').sum():,}")
print(f"  intersection     : {(targets['_pattern']=='intersection').sum():,}")
print(f"\nAnchor breakdown:")
print(targets["vb_anchor"].value_counts().to_string())

#### 3.3 Census Geocoding

##### 3.3.1 Geocoding function 

In [ ]:
# Census batch geocoder function

def submit_census_batch(batch_df, retries=3):
    lines = []
    for idx, row in batch_df.iterrows():
        street = str(row["census_street"]).replace(",", " ").replace('"', "")
        city   = str(row["census_city"]).replace(",", " ").replace('"', "")
        zipc   = str(row["census_zip"]) if str(row["census_zip"]) != "nan" else ""
        lines.append(f'{idx},"{street}","{city}","CA","{zipc}"')

    csv_bytes = "\n".join(lines).encode("utf-8")

    for attempt in range(retries):
        try:
            http = urllib3.PoolManager()
            resp = http.request(
                "POST",
                CENSUS_URL,
                fields={
                    "addressFile": ("batch.csv", csv_bytes, "text/plain"),
                    "benchmark":   "Public_AR_Current",
                },
                timeout=120,
            )
            if resp.status != 200:
                print(f" HTTP {resp.status}")
                time.sleep(15 * (attempt + 1))
                continue

            raw = resp.data.decode("latin-1")
            out = []
            for row in csv.reader(io.StringIO(raw)):
                if len(row) < 4:
                    continue
                try:
                    lon, lat = [float(x) for x in row[5].strip().split(",")]
                except:
                    lat = lon = None
                out.append({
                    "orig_idx":   row[0].strip(),
                    "match":      row[2].strip(),
                    "match_type": row[3].strip() if len(row) > 3 else "",
                    "geo_lat":    lat,
                    "geo_lon":    lon,
                })
            return out

        except Exception as e:
            print(f" Attempt {attempt+1} failed: {e}")
            time.sleep(15 * (attempt + 1))

    return []

##### 3.3.2 Geocoding Loop

In [ ]:
# Geocoding loop Run with checkpoint 
completed = set()
if os.path.exists(CENSUS_CHECKPOINT_FILE):
    ex = pd.read_csv(CENSUS_CHECKPOINT_FILE, dtype=str)
    completed = set(ex["orig_idx"])
    print(f"Resuming - {len(completed):,} done.")
else:
    with open(CENSUS_CHECKPOINT_FILE, "w") as f:
        f.write("orig_idx,match,match_type,geo_lat,geo_lon\n")
    print("Starting fresh.")

remaining = targets[~targets.index.astype(str).isin(completed)]
batches   = [remaining.iloc[i:i+CENSUS_BATCH_SIZE]
             for i in range(0, len(remaining), CENSUS_BATCH_SIZE)]
print(f"Remaining: {len(remaining):,}  |  Batches: {len(batches)}\n")

for i, batch in enumerate(batches):
    print(f"Batch {i+1}/{len(batches)} ({len(batch):,})", end=" ... ")
    results = submit_census_batch(batch)
    if not results:
        print("FAILED"); continue
    with open(CENSUS_CHECKPOINT_FILE, "a") as f:
        rmap = {r["orig_idx"]: r for r in results}
        for idx in batch.index:
            r = rmap.get(str(idx), {})
            f.write(f"{idx},{r.get('match','No_Match')},"
                    f"{r.get('match_type','')},"
                    f"{r.get('geo_lat','')},"
                    f"{r.get('geo_lon','')}\n")
    matched = sum(1 for r in results if r.get("match") == "Match")
    print(f"{matched}/{len(results)}")

##### 3.3.3 Load & report

In [ ]:
# Load results, filter OOB, report

res = pd.read_csv(CENSUS_CHECKPOINT_FILE, dtype=str)
res["orig_idx"] = res["orig_idx"].astype(str)
res = res.drop_duplicates("orig_idx", keep="last")
res["geo_lat"] = pd.to_numeric(res["geo_lat"], errors="coerce")
res["geo_lon"] = pd.to_numeric(res["geo_lon"], errors="coerce")

oob = ((res["geo_lat"] < LAT_MIN) | (res["geo_lat"] > LAT_MAX) |
       (res["geo_lon"] < LON_MIN) | (res["geo_lon"] > LON_MAX))
res.loc[oob, ["geo_lat","geo_lon"]] = None
res.loc[oob, "match"] = "Out_Of_Region"

matched_res = res[res["match"] == "Match"]
geocoded_idx = set(matched_res["orig_idx"])

print("Census Geocoding Final Coverage Report")
print(f"{'Year':<8} {'Total':>8}  {'Stored':>8}  {'Geocoded':>9}  {'Junk':>7}  {'Unresolved':>11}  {'Cov%':>6}")

for yr in sorted(df["data_year"].unique()):
    sub = df[df["data_year"] == yr]
    n = len(sub)
    stored = (sub["_bucket"] == "A").sum()
    yr_tgt = targets[targets["data_year"] == yr] if "data_year" in targets.columns else pd.DataFrame()
    yr_idx = set(yr_tgt.index.astype(str))
    geocoded = len(matched_res[matched_res["orig_idx"].isin(yr_idx)])
    junk = int(yr_tgt["_pattern"].isna().sum()) if len(yr_tgt) else 0
    unres = n - stored - geocoded
    print(f"{yr:<8} {n:>8,}  {stored:>8,}  {geocoded:>9,}  {junk:>7,}  {max(unres,0):>11,}  {(stored+geocoded)/n*100:>5.1f}%")


print("Unresolved Breakdown")
unres_df = targets[~targets.index.astype(str).isin(geocoded_idx)]
for pat, c in unres_df["_pattern"].value_counts().items():
    print(f"{pat}: {c}")
# Intersection and numbered addresses that fail here go to Nominatim below

#### 3.4 Nominatim Geocoding

##### 3.4.1 Geocodig Function

In [ ]:
# Nominatim fallback for Census failures


# Build the fallback set: unresolved rows from Census run
geocoded_idx = set(
    pd.read_csv(CENSUS_CHECKPOINT_FILE, dtype=str)
    .query("match == 'Match'")["orig_idx"]
)
fallback = targets[~targets.index.astype(str).isin(geocoded_idx)].copy()


print(f"Filtered fallback : {len(fallback):,}")
print(f"Nominatim fallback targets : {len(fallback):,}")
print(f"intersection : {(fallback['_pattern']=='intersection').sum():,}")
print(f"numbered : {(fallback['_pattern']=='numbered').sum():,}")


def build_nominatim_query(row):
    """
    Returns query string and viewbox string for Nominatim.
    Intersection: 'STREET_A & CROSS_STREET, city, CA zip'
    Numbered: '123 MAIN ST, city, CA zip'
    Both use viewbox+bounded to constrain to the known district area
    """
    street = str(row["census_street"])
    city   = str(row["census_city"])
    zipc   = str(row["census_zip"])

    if zipc and zipc != "nan" and len(zipc) == 5:
        q = f"{street}, {city}, CA {zipc}"
    else:
        q = f"{street}, {city}, CA"

    # Viewbox: lon_min,lat_min,lon_max,lat_max
    lat   = row["vb_lat"]
    lon   = row["vb_lon"]
    delta = row["vb_delta"]
    vbox  = f"{lon-delta:.6f},{lat-delta:.6f},{lon+delta:.6f},{lat+delta:.6f}"
    return q, vbox


def nominatim_geocode(query, viewbox):
    params = {"q":query,"format":"jsonv2","addressdetails": 0,"limit":1,"countrycodes": "us","viewbox":viewbox,"bounded":1}
    req = urllib.request.Request(f"{NOMINATIM_URL}?{urllib.parse.urlencode(params)}", headers={"User-Agent": USER_AGENT}
                                 )
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            results = json.loads(resp.read().decode())
        if not results:
            return None, None, "no_result"
        r = results[0]
        return float(r["lat"]), float(r["lon"]), r.get("type", "unknown")
    except Exception as e:
        return None, None, f"error:{e}"

##### 3.4.2 Geocoding Loop

In [ ]:
# Geocoding loop Run with checkpoint
nom_completed = set()
if os.path.exists(NOMINATIM_CHECKPOINT_FILE):
    ex = pd.read_csv(NOMINATIM_CHECKPOINT_FILE, dtype=str)
    nom_completed = set(ex["orig_idx"])
    print(f"Resuming Nominatim - {len(nom_completed):,} done.")
else:
    with open(NOMINATIM_CHECKPOINT_FILE, "w") as f:
        f.write("orig_idx,match,geo_lat,geo_lon\n")
    print("Starting Nominatim fresh")

nom_remaining = fallback[~fallback.index.astype(str).isin(nom_completed)]
print(f"Remaining : {len(nom_remaining):,}")

for i, (idx, row) in tqdm(enumerate(nom_remaining.iterrows()), total=len(nom_remaining)):

    query, viewbox = build_nominatim_query(row)
    lat, lon, qual = nominatim_geocode(query, viewbox)
    
    # OOB check
    if lat is not None:
        if not (LAT_MIN <= lat <= LAT_MAX and LON_MIN <= lon <= LON_MAX):
            lat = lon = None
            qual = "out_of_region"

    match = "Match" if lat is not None else "No_Match"

    with open(NOMINATIM_CHECKPOINT_FILE, "a") as f:
        f.write(f"{idx},{match},"
                f"{lat if lat is not None else ''},"
                f"{lon if lon is not None else ''}\n")

    time.sleep(DELAY_SEC)

##### 3.4.2 Load & Report

In [ ]:
# Load and report

nom_res = pd.read_csv(NOMINATIM_CHECKPOINT_FILE, dtype=str)
nom_res["orig_idx"] = nom_res["orig_idx"].astype(str)
nom_res = nom_res.drop_duplicates("orig_idx", keep="last")
nom_res["geo_lat"] = pd.to_numeric(nom_res["geo_lat"], errors="coerce")
nom_res["geo_lon"] = pd.to_numeric(nom_res["geo_lon"], errors="coerce")

nom_matched = nom_res[nom_res["match"] == "Match"]

print("Nominatim Geocoding Final Coverage Report")
print(f"Sent : {len(nom_res):,}")
print(f"Matched : {len(nom_matched):,} ")
print(f"No result : {(nom_res['match']=='No_Match').sum():,}")


print("By Pattern : ")
for pat in ["numbered","intersection"]:
    sub = fallback[fallback["_pattern"] == pat]
    sidx = set(sub.index.astype(str))
    nm = nom_matched[nom_matched["orig_idx"].isin(sidx)]
    print(f"{pat:<15s} sent={len(sub):>8,}  matched={len(nm):>8,}  ({len(nm)/len(sub)*100:.1f}%)")

### Phase 4 Output file

#### 4.0 Geocoding Results into Preprocessed DF

In [ ]:
# Merge geocoding results into df

def load_geocode_results(path, oob_filter=True):
    res = pd.read_csv(path, dtype=str)
    res["orig_idx"] = res["orig_idx"].astype(str)
    res = res.drop_duplicates("orig_idx", keep="last")
    res["geo_lat"] = pd.to_numeric(res["geo_lat"], errors="coerce")
    res["geo_lon"] = pd.to_numeric(res["geo_lon"], errors="coerce")
    if oob_filter:
        oob = ((res["geo_lat"] < LAT_MIN) | (res["geo_lat"] > LAT_MAX) |
               (res["geo_lon"] < LON_MIN) | (res["geo_lon"] > LON_MAX))
        res.loc[oob, ["geo_lat","geo_lon"]] = None
        res.loc[oob, "match"] = "Out_Of_Region"
    return res[res["match"] == "Match"][["orig_idx","geo_lat","geo_lon"]]

census_matched    = load_geocode_results(CENSUS_CHECKPOINT_FILE)
nominatim_matched = load_geocode_results(NOMINATIM_CHECKPOINT_FILE)

print(f"Census matched : {len(census_matched):,}")
print(f"Nominatim matched : {len(nominatim_matched):,}")

# Census takes priority - only use Nominatim where Census has no result
census_idx = set(census_matched["orig_idx"])
nom_new = nominatim_matched[~nominatim_matched["orig_idx"].isin(census_idx)]
print(f"Nominatim new (not already in Census): {len(nom_new):,}")

all_geocoded = pd.concat([census_matched, nom_new], ignore_index=True)
all_geocoded = all_geocoded.set_index("orig_idx")


# Assign lat_final, lon_final, coord_source
df["lat_final"] = None
df["lon_final"] = None
df["coord_source"] = "unresolved"

# Bucket A - stored coords
bucket_a = df["_bucket"] == "A"
df.loc[bucket_a, "coord_source"] = "stored"
df.loc[bucket_a, "lat_final"] = df.loc[bucket_a, "LAT_f"]
df.loc[bucket_a, "lon_final"] = df.loc[bucket_a, "LON_f"]

# Geocoded rows (Census + Nominatim)
geo_idx = all_geocoded.index.intersection(df.index.astype(str))
df_idx  = df.index.astype(str).isin(geo_idx)

df.loc[df_idx, "lat_final"] = all_geocoded.loc[df.index[df_idx].astype(str), "geo_lat"].values
df.loc[df_idx, "lon_final"] = all_geocoded.loc[df.index[df_idx].astype(str), "geo_lon"].values

# Label source: census vs nominatim
census_df_idx = df.index.astype(str).isin(census_idx)
nominatim_df_idx = df.index.astype(str).isin(set(nom_new["orig_idx"]))
df.loc[census_df_idx, "coord_source"] = "census"
df.loc[nominatim_df_idx, "coord_source"] = "nominatim"

print(f"coord_source breakdown:")
print(df["coord_source"].value_counts().to_string())

# Write output: Drop all internal/working columns and keep only original + 3 derived cols
internal = {c for c in df.columns if c.startswith("_")}
working  = {
    "LAT_f","LON_f",
    "vb_lat","vb_lon","vb_delta","vb_str","vb_anchor",
    "fwd_lat","fwd_lon","fwd_dist_m","fwd_quality",
    "fwd_address","fwd_pattern","coord_conflict",
    "rev_road","rev_display","rev_postcode","rev_city",
    "rev_error","rev_house_number","rev_suburb",
    "match_street","match_zip","coord_verified",
    "coord_suspect","coord_verdict","dist_bin",
    "census_street","census_city","census_state","census_zip",
}

keep = [c for c in df.columns if c not in internal and c not in working]
keep += ["lat_final", "lon_final", "coord_source"]
keep  = list(dict.fromkeys(keep))   # deduplicate, preserve order

out_df = df[[c for c in keep if c in df.columns]].copy()
out_df.to_csv(PREPROCESSED, index=False)

print(f"Rows : {len(out_df):,}")
print(f"Columns : {len(out_df.columns)}")

#### 4.2 Coverage Report

In [ ]:
# Coverage report

n   = len(df)
print("Final Coverage Report")

for src, label in [("stored","Stored coords, clean"),("census","Census geocoded"),("nominatim","Nominatim geocoded"),("unresolved","No coordinates")]:
    c = (df["coord_source"] == src).sum()
    print(f"{label}: {c}  ({c/n*100:.1f}%)")

has = df["lat_final"].notna().sum()
print(f"\nTotal WITH Coords : {has:,}  ({has/n*100:.1f}%)")
print(f"Total WITHOUT Coords : {n-has:,}  ({(n-has)/n*100:.1f}%)")

print(f"  {'Year':<8} {'Total':>8}  {'WithCoord':>10}  {'Cov%':>6}")
for yr in sorted(df["data_year"].unique()):
    sub = df[df["data_year"] == yr]
    has_c = sub["lat_final"].notna().sum()
    print(f"  {yr:<8} {len(sub):>8,}  {has_c:>10,}  {has_c/len(sub)*100:>5.1f}%")